In [ ]:
#Dendrogram
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster, cut_tree

Z = linkage(X_final, method='ward')

clusters_30 = fcluster(Z, t=30, criterion='maxclust')

plt.figure(figsize=(16, 6))
plt.title("Dendrogram with Cluster Cut (k=30)")
plt.xlabel("Sample index")
plt.ylabel("Distance")

dendrogram(
    Z,
    truncate_mode='lastp',
    p=30,
    show_leaf_counts=True,
    leaf_rotation=90.,
    leaf_font_size=10.,
    color_threshold=None
)

plt.axhline(y=Z[-25, 2], color='red', linestyle='--', label='k=30 cut')
plt.legend()
plt.tight_layout()
plt.show()

In [1]:
#Bubble plot illustrating residue composition across 30 clusters
from scipy.cluster.hierarchy import fcluster
import seaborn as sns
import matplotlib.pyplot as plt

num_clusters = 30
df['Cluster_Label'] = fcluster(Z, num_clusters, criterion='maxclust')

residue_count_cols = [
    'ARG_Count', 'ASN_Count', 'ASP_Count', 'CYS_Count',
    'GLN_Count', 'GLU_Count', 'GLY_Count', 'HIS_Count',
    'MET_Count', 'SER_Count', 'THR_Count', 'TYR_Count',
    'VAL_Count', 'Water_Count','PHE_Count', 'ILE_Count',
    'LEU_Count', 'ALA_Count', 'LYS_Count','TRP_Count','PRO_Count',
]

def get_residue_combination(row):
    return ' '.join([
        f"{int(row[col])} {col.split('_')[0]}" for col in residue_count_cols
        if col in row and row[col] > 0
    ])

df['Residue_Combination'] = df.apply(get_residue_combination, axis=1)

combo_counts = df.groupby(['Cluster_Label', 'Residue_Combination']).size().reset_index(name='Site_Count')

unique_clusters = sorted(combo_counts['Cluster_Label'].unique())
cluster_palette = sns.color_palette("pastel", len(unique_clusters))
cluster_color_map = dict(zip(unique_clusters, cluster_palette))
combo_counts['Color'] = combo_counts['Cluster_Label'].map(cluster_color_map)

print("\n=== Cluster Sizes ===")
cluster_sizes = df['Cluster_Label'].value_counts().sort_index()
for cluster_id, size in cluster_sizes.items():
    print(f"Cluster {cluster_id}: {size} sites")

clusters_per_plot = 5
for start in range(1, num_clusters + 1, clusters_per_plot):
    end = start + clusters_per_plot - 1
    cluster_range = list(range(start, min(end + 1, num_clusters + 1)))
    data_subset = combo_counts[combo_counts['Cluster_Label'].isin(cluster_range)]

    plt.figure(figsize=(10, 30))
    plot = sns.scatterplot(
        data=data_subset,
        x='Cluster_Label', y='Residue_Combination', size='Site_Count',
        hue='Cluster_Label', palette=cluster_color_map,
        sizes=(100, 2000), alpha=0.85, legend=False
    )

    for line in range(data_subset.shape[0]):
        x = data_subset.Cluster_Label.iloc[line]
        y = data_subset.Residue_Combination.iloc[line]
        count = data_subset.Site_Count.iloc[line]
        plot.text(
            x, y, str(count),
            ha='center', va='center', weight='bold', fontsize=9, color='black'
        )

    plt.title(f'Bubble Plot: Clusters {start} to {end}', fontsize=16, fontweight='bold')
    plt.xlabel('Cluster Label', fontsize=14)
    plt.ylabel('Residue Combination', fontsize=14)
    plt.xticks(cluster_range)
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [ ]:
#Residue composition deviation (RCD) score analysis
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

cluster_label_map = {
    1: "T1_C1_Cys-Met",   2: "T1_C3_Cys-Gly",   3: "T1_C2_Cys-Gln",
    4: "T1_C4_Cys-His",   5: "T1_C4_Cys-Glu",   6: "T1_C4_Cys",
    7: "T2_C1_His-A",    8: "T2_C1_His-B",
    9: "T2_C2_Ile",     10: "T2_C2_Lys",     11: "T2_C2_His-HOH-A",
    12: "T2_C2_His-HOH-B", 13: "T2_C2_Gln", 14: "T2_C2_Arg", 15: "T2_C2_Gly",
    16: "T2_C3_Met-His", 17: "T2_C3_Met-X",
    18: "T2_C4_O-A",    19: "T2_C4_O-B",
    20: "T2_C5_ArA",    21: "T2_C5_ArB",
    22: "T2_C6_Thr-Val", 23: "T2_C6_Thr-Cys", 24: "T2_C6_Thr/Val",
    25: "T2_C7_Asn",    26: "T2_C7_Ser",
    27: "T2_C8_Ala",
    28: "T2_C9_Asp",    29: "T2_C9_Glu-A",  30: "T2_C9_Glu-B"
}

selected_features = [
    'ARG_Count', 'ASN_Count', 'ASP_Count', 'CYS_Count',
    'GLN_Count', 'GLU_Count', 'GLY_Count', 'HIS_Count',
    'MET_Count', 'SER_Count', 'THR_Count', 'TYR_Count',
    'VAL_Count', 'PHE_Count', 'ILE_Count',
    'LEU_Count', 'ALA_Count', 'LYS_Count', 'TRP_Count'
]

tab20 = plt.get_cmap("tab20").colors

residue_list = [
    'ARG','ASN','ASP','CYS','GLN','GLU','GLY','HIS','MET',
    'SER','THR','TYR','VAL','PHE','ILE','LEU','ALA','LYS',
    'TRP'
]

residue_colors = {
    res: tab20[i % len(tab20)]
    for i, res in enumerate(residue_list)
}

TOP_N = 3

df_feat = df[selected_features].copy()
df_feat["cluster"] = clusters_30

cluster_means = (
    df_feat
    .groupby("cluster")[selected_features]
    .mean()
)

mean_1_6 = cluster_means.loc[1:6].mean()
mean_7_30 = cluster_means.loc[7:30].mean()

rows = []

for k in range(1, 31):

    cluster_mean = (
        df_feat[
            df_feat["cluster"] == k
        ][selected_features].mean()
    )

    reference_mean = (
        mean_1_6 if k <= 6 else mean_7_30
    )

    importance = cluster_mean - reference_mean

    top_features = (
        importance.abs()
        .sort_values(ascending=False)
        .head(TOP_N)
        .index
    )

    for feat in top_features:

        rows.append({
            "Cluster": k,
            "Residue": feat.replace("_Count", ""),
            "Importance": importance[feat]
        })

plot_df = pd.DataFrame(rows)

fig, (ax1, ax2) = plt.subplots(
    2, 1,
    figsize=(16, 18),
    gridspec_kw={"height_ratios": [1, 4]},
    sharex=True
)

def plot_cluster_range(ax, cluster_range, title):

    for k in cluster_range:

        cluster_data = plot_df[
            plot_df["Cluster"] == k
        ]

        left_pos = 0

        for _, row in cluster_data.iterrows():

            value = row["Importance"]

            if value <= 0:
                continue

            ax.barh(
                k,
                value,
                left=left_pos,
                color=residue_colors[row["Residue"]],
                edgecolor='black',
                linewidth=2,
                height=0.7
            )

            left_pos += value

    ax.axvline(
        0,
        color='black',
        linewidth=3
    )

    ax.set_yticks(list(cluster_range))

    ax.set_yticklabels(
        [
            cluster_label_map[i]
            for i in cluster_range
        ],
        fontsize=18,
        fontweight='bold'
    )

    ax.set_title(
        title,
        fontsize=22,
        fontweight='bold'
    )

    ax.invert_yaxis()

    for spine in ax.spines.values():
        spine.set_linewidth(3)

plot_cluster_range(
    ax1,
    range(1, 7),
    "Clusters 1–6 (Type-1 motifs)"
)

plot_cluster_range(
    ax2,
    range(7, 31),
    "Clusters 7–30 (Type-2 motifs)"
)

ax2.set_xlabel(
    "Cluster Mean − Reference Mean",
    fontsize=22,
    fontweight='bold'
)

for ax in [ax1, ax2]:

    ax.tick_params(
        axis='x',
        labelsize=18,
        width=3,
        length=8
    )

    for label in ax.get_xticklabels():
        label.set_fontweight('bold')

t1_residues = [
    'MET',
    'GLY',
    'HIS',
    'GLU',
    'GLN',
    'CYS'
]

t1_handles = [
    plt.Line2D(
        [0],
        [0],
        color=residue_colors[r],
        lw=8
    )
    for r in t1_residues
]

ax1.legend(
    t1_handles,
    t1_residues,
    title="Residues (Type-1)",
    title_fontsize=18,
    fontsize=16,
    prop={'weight': 'bold'},
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=True
)

t2_handles = [
    plt.Line2D(
        [0],
        [0],
        color=residue_colors[r],
        lw=8
    )
    for r in residue_list
]

ax2.legend(
    t2_handles,
    residue_list,
    title="Residues (Type-2)",
    title_fontsize=18,
    fontsize=14,
    prop={'weight': 'bold'},
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=True
)

plt.tight_layout()

plt.savefig(
    "RCD-score.png",
    dpi=300,
    bbox_inches='tight'
)

plt.show()

In [ ]:
# Mean feature value plot
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import numpy as np
from scipy.cluster.hierarchy import fcluster
from matplotlib.colors import LinearSegmentedColormap, Normalize, TwoSlopeNorm

num_clusters = 30

df['Cluster_Label'] = fcluster(
    Z,
    num_clusters,
    criterion='maxclust'
)

df['Cluster_Name'] = df['Cluster_Label'].map(cluster_label_map)

cluster_summary = (
    df.groupby('Cluster_Name')[numerical_cols]
    .mean()
)

cluster_summary = cluster_summary.reindex(
    cluster_label_map.values()
)

data = cluster_summary.T

group1 = [
    'ARG_Count',
    'ASN_Count',
    'ASP_Count',
    'CYS_Count',
    'GLN_Count',
    'GLU_Count',
    'GLY_Count',
    'HIS_Count',
    'MET_Count',
    'SER_Count',
    'THR_Count',
    'TYR_Count',
    'VAL_Count',
    'PHE_Count',
    'ILE_Count',
    'LEU_Count',
    'ALA_Count',
    'LYS_Count',
    'TRP_Count'
]

group2 = [
    'Comb-Atom_N',
    'Comb-Atom_S',
    'Comb-Atom_O'
]

group3 = [
    'Class_Nature_Non-polar',
    'Class_Nature_Polar acidic',
    'Class_Nature_Polar basic',
    'Class_Nature_Polar neutral',
    'Class_Nature_Polar O',
    'Aromatic_Count'
]

group4 = [
    'Average_Isoelectric_Point'
]

group5 = [
    'Average_Hydrophobicity'
]

def make_mask(rows):
    return np.tile(
        ~data.index.isin(rows)[:, None],
        (1, data.shape[1])
    )

mask_g1 = make_mask(group1)
mask_g2 = make_mask(group2)
mask_g3 = make_mask(group3)
mask_g4 = make_mask(group4)
mask_g5 = make_mask(group5)

cmap_g1 = LinearSegmentedColormap.from_list(
    "aa_green_rev",
    [
        "#ffffff",
        "#d9f0d3",
        "#41ab5d",
        "#005a32"
    ]
)

cmap_g2 = LinearSegmentedColormap.from_list(
    "atom_purple_rev",
    [
        "#ffffff",
        "#dadaeb",
        "#9e9ac8",
        "#54278f"
    ]
)

cmap_g3 = LinearSegmentedColormap.from_list(
    "class_orange_rev",
    [
        "#ffffff",
        "#fdd0a2",
        "#fd8d3c",
        "#7f2704"
    ]
)

def smart_norm(group):

    vals = data.loc[
        data.index.isin(group)
    ]

    vmin = vals.min().min()
    vmax = vals.max().max()

    if vmin < 0 and vmax > 0:
        return TwoSlopeNorm(
            vmin=vmin,
            vcenter=0,
            vmax=vmax
        )
    else:
        return Normalize(
            vmin=vmin,
            vmax=vmax
        )

norm_g1 = smart_norm(group1)
norm_g2 = smart_norm(group2)
norm_g3 = smart_norm(group3)
norm_g4 = smart_norm(group4)
norm_g5 = smart_norm(group5)

fig, ax = plt.subplots(
    figsize=(25, 20),
    dpi=300
)

sns.heatmap(
    data,
    cmap=cmap_g1,
    norm=norm_g1,
    mask=mask_g1,
    annot=True,
    fmt='.1f',
    linewidths=1.8,
    linecolor='black',
    annot_kws={
        "size": 18,
        "weight": "bold",
        "color": "black"
    },
    cbar=False,
    ax=ax
)

sns.heatmap(
    data,
    cmap=cmap_g2,
    norm=norm_g2,
    mask=mask_g2,
    annot=True,
    fmt='.1f',
    linewidths=1.8,
    linecolor='black',
    annot_kws={
        "size": 18,
        "weight": "bold",
        "color": "black"
    },
    cbar=False,
    ax=ax
)

sns.heatmap(
    data,
    cmap=cmap_g3,
    norm=norm_g3,
    mask=mask_g3,
    annot=True,
    fmt='.1f',
    linewidths=1.8,
    linecolor='black',
    annot_kws={
        "size": 18,
        "weight": "bold",
        "color": "black"
    },
    cbar=False,
    ax=ax
)

sns.heatmap(
    data,
    cmap='coolwarm',
    norm=norm_g4,
    mask=mask_g4,
    annot=True,
    fmt='.2f',
    linewidths=1.8,
    linecolor='black',
    annot_kws={
        "size": 18,
        "weight": "bold",
        "color": "black"
    },
    cbar=False,
    ax=ax
)

sns.heatmap(
    data,
    cmap='RdYlBu_r',
    norm=norm_g5,
    mask=mask_g5,
    annot=True,
    fmt='.2f',
    linewidths=1.8,
    linecolor='black',
    annot_kws={
        "size": 18,
        "weight": "bold",
        "color": "black"
    },
    cbar=False,
    ax=ax
)

ax.set_xlabel(
    'Clusters',
    fontsize=40,
    fontweight='bold',
    labelpad=35
)

ax.set_ylabel(
    'Features',
    fontsize=40,
    fontweight='bold',
    labelpad=20
)

ax.set_xticklabels(
    ax.get_xticklabels(),
    rotation=90,
    fontsize=28,
    fontweight='bold',
    color='black'
)

ax.set_yticklabels(
    ax.get_yticklabels(),
    fontsize=28,
    fontweight='bold',
    color='black'
)

for spine in ax.spines.values():
    spine.set_linewidth(3.5)
    spine.set_color("black")

ax.tick_params(
    axis='both',
    width=3,
    length=8,
    colors='black'
)

plt.tight_layout()

plt.savefig(
    "Mean-feature-value.png",
    dpi=300,
    bbox_inches='tight'
)

plt.show()

fig = plt.figure(
    figsize=(15, 8),
    dpi=300
)

bars = [
    (
        cmap_g1,
        norm_g1,
        "Liganding residue counts",
        0.75
    ),
    (
        cmap_g2,
        norm_g2,
        "Liganding atom counts",
        0.58
    ),
    (
        cmap_g3,
        norm_g3,
        "Nature of liganding residue count",
        0.41
    ),
    (
        mpl.cm.coolwarm,
        norm_g4,
        "Average Isoelectric Point",
        0.24
    ),
    (
        mpl.cm.RdYlBu_r,
        norm_g5,
        "Average Hydrophobicity",
        0.07
    )
]

for cmap, norm, label, ypos in bars:

    ax_bar = fig.add_axes(
        [0.15, ypos, 0.7, 0.06]
    )

    mpl.colorbar.ColorbarBase(
        ax_bar,
        cmap=cmap,
        norm=norm,
        orientation='horizontal'
    )

    ax_bar.tick_params(
        labelsize=18
    )

    for tick in ax_bar.get_xticklabels():
        tick.set_fontweight('bold')

    ax_bar.set_title(
        label,
        fontsize=20,
        fontweight='bold',
        pad=10
    )

plt.savefig(
    "Mean-feature-value-Legends.png",
    dpi=300,
    bbox_inches='tight'
)

plt.show()

In [ ]:
#Annotated dendogram
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as colors
from scipy.cluster.hierarchy import dendrogram


# Cluster labels
cluster_label_map = {
    1: "T1_C1_Cys-Met",
    2: "T1_C3_Cys-Gly",
    3: "T1_C2_Cys-Gln",
    4: "T1_C4_Cys-His",
    5: "T1_C4_Cys-Glu",
    6: "T1_C4_Cys",

    7: "T2_C1_His-A",
    8: "T2_C1_His-B",
    9: "T2_C2_Ile",
    10: "T2_C2_Lys",
    11: "T2_C2_His-HOH-A",
    12: "T2_C2_His-HOH-B",
    13: "T2_C2_Gln",
    14: "T2_C2_Arg",
    15: "T2_C2_Gly",

    16: "T2_C3_Met-His",
    17: "T2_C3_Met-X",

    18: "T2_C4_O-A",
    19: "T2_C4_O-B",

    20: "T2_C5_ArA",
    21: "T2_C5_ArB",

    22: "T2_C6_Thr-Val",
    23: "T2_C6_Thr-Cys",
    24: "T2_C6_Thr/Val",

    25: "T2_C7_Asn",
    26: "T2_C7_Ser",

    27: "T2_C8_Ala",

    28: "T2_C9_Asp",
    29: "T2_C9_Glu-A",
    30: "T2_C9_Glu-B"
}



# Site counts
cluster_sitecount_map = {
    1: 770,
    2: 179,
    3: 32,
    4: 771,
    5: 32,
    6: 102,

    7: 199,
    8: 881,

    9: 20,
    10: 27,
    11: 436,
    12: 420,
    13: 63,
    14: 20,
    15: 36,

    16: 71,
    17: 71,

    18: 9,
    19: 4,

    20: 2,
    21: 103,

    22: 8,
    23: 7,
    24: 7,

    25: 47,
    26: 40,

    27: 45,

    28: 247,
    29: 75,
    30: 168
}


# STRONGLY SEPARATED CLASS COLORS

# ---- TYPE 1: ORANGE ----
orange_shades = [
    "#f16913",
    "#d94801",
    "#c24a02",
    "#7f2704"
]


# ---- TYPE 2: GREEN ----
green_shades = [
    "#2e7d32",  # C1
    "#1b5e20",  # C2
    "#388e3c",  # C3
    "#004d40",  # C4
    "#33691e",  # C5
    "#00695c",  # C6
    "#43a047",  # C7
    "#1f7a1f",  # C8
    "#0b3d0b"   # C9
]



# Class color map
class_color_map = {

    # -------- TYPE 1 --------
    1: "#f16913",
    2: "#d94801",
    3: "#c24a02",
    4: "#7f2704",
    5: "#7f2704",
    6: "#7f2704",

    # -------- TYPE 2 --------
    7: green_shades[0],
    8: green_shades[0],

    9: green_shades[1],
    10: green_shades[1],
    11: green_shades[1],
    12: green_shades[1],
    13: green_shades[1],
    14: green_shades[1],
    15: green_shades[1],

    16: green_shades[2],
    17: green_shades[2],

    18: green_shades[3],
    19: green_shades[3],

    20: green_shades[4],
    21: green_shades[4],

    22: green_shades[5],
    23: green_shades[5],
    24: green_shades[5],

    25: green_shades[6],
    26: green_shades[6],

    27: green_shades[7],

    28: green_shades[8],
    29: green_shades[8],
    30: green_shades[8]
}



# FIGURE & AXES
fig = plt.figure(figsize=(8, 16), dpi=300)

ax_d = fig.add_axes(
    [0.08, 0.05, 0.68, 0.9],
    zorder=3
)

ax_b = fig.add_axes(
    [0.76, 0.05, 0.22, 0.9],
    sharey=ax_d,
    zorder=1
)

ax_b.patch.set_visible(False)

# DENDROGRAM
dendrogram(
    Z,
    truncate_mode="lastp",
    p=30,
    orientation="left",
    show_leaf_counts=False,
    color_threshold=None,
    ax=ax_d
)

ax_d.invert_yaxis()

for coll in ax_d.collections:
    coll.set_linewidth(4)

ax_d.set_xlabel(
    "Distance",
    fontsize=32,
    fontweight="bold"
)

ax_d.tick_params(
    axis="x",
    labelsize=26,
    width=4,
    length=10
)

for t in ax_d.get_xticklabels():
    t.set_fontweight("bold")

ax_d.yaxis.set_visible(False)

for s in ax_d.spines.values():
    s.set_linewidth(3)

# BUBBLE SIZES
leaf_y = ax_d.get_yticks()

leaf_order = list(range(1, 31))

counts = np.array([
    cluster_sitecount_map[i]
    for i in leaf_order
])

bubble_sizes = np.interp(
    counts,
    (counts.min(), counts.max()),
    (180, 1200)
)

# BUBBLE COLOR MAPS
orange = cm.get_cmap("Oranges")
green = cm.get_cmap("Greens")


def oc(v):
    return orange(0.45 + 0.55 * v)


def gc(v):
    return green(0.45 + 0.55 * v)

# NORMALIZATION
norm_o = colors.Normalize(
    min(cluster_sitecount_map[i] for i in range(1, 7)),
    max(cluster_sitecount_map[i] for i in range(1, 7))
)

norm_g = colors.Normalize(
    min(cluster_sitecount_map[i] for i in range(7, 31)),
    max(cluster_sitecount_map[i] for i in range(7, 31))
)

# BUBBLE AXIS STYLING
ax_b.set_xlim(0, 1)
ax_b.set_xticks([])
ax_b.yaxis.set_visible(False)

for s in ax_b.spines.values():
    s.set_visible(False)

# DRAW CLUSTER BUBBLES
for y, idx, size, count in zip(
    leaf_y,
    leaf_order,
    bubble_sizes,
    counts
):

    # Select bubble color according to Type 1 / Type 2
    if idx <= 6:
        bubble_color = oc(norm_o(count))
    else:
        bubble_color = gc(norm_g(count))

    # Dotted connector between dendrogram and bubble
    ax_b.plot(
        [0.0, 0.38],
        [y, y],
        linestyle=":",
        linewidth=2.5,
        color="black"
    )

    # Site count
    ax_b.text(
        0.15,
        y + 0.35,
        str(count),
        ha="center",
        va="bottom",
        fontsize=15,
        fontweight="bold"
    )

    # Bubble
    ax_b.scatter(
        0.38,
        y,
        s=size,
        color=bubble_color,
        edgecolor="black",
        linewidth=1.4,
        zorder=5
    )

    # Cluster name
    ax_b.text(
        0.55,
        y,
        cluster_label_map[idx],
        va="center",
        ha="left",
        fontsize=25,
        fontweight="bold",
        color=class_color_map[idx]
    )

#SAVE FIGURE
plt.savefig(
    "Dendrogram-Figure3.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
#tSNE
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
import matplotlib as mpl
from matplotlib.lines import Line2D

mpl.rcParams["font.weight"] = "bold"
mpl.rcParams["axes.labelweight"] = "bold"

tsne = TSNE(
    n_components=2,
    perplexity=30,
    learning_rate='auto',
    init='pca',
    random_state=42
)

X_tsne = tsne.fit_transform(X_final)

orange_shades = [
    "#fff3b0",
    "#ffb703",
    "#fb8500",
    "#e85d04",
    "#d00000",
    "#7f2704"
]

green_base_shades = [
    "#b9f6ca",
    "#69f0ae",
    "#00e676",
    "#00c853",
    "#2e7d32",
    "#1b5e20",
    "#00bfa5",
    "#00897b"
]

markers = ['o', '^', 'x']

def style_axes():

    plt.xlabel(
        "t-SNE 1",
        fontsize=25,
        fontweight="bold"
    )

    plt.ylabel(
        "t-SNE 2",
        fontsize=25,
        fontweight="bold"
    )

    plt.xticks(
        fontsize=20,
        fontweight="bold"
    )

    plt.yticks(
        fontsize=20,
        fontweight="bold"
    )

    ax = plt.gca()

    for spine in ax.spines.values():
        spine.set_linewidth(2)

    plt.tight_layout()

plt.figure(figsize=(8, 8))

for cluster in range(1, 7):

    mask = clusters_30 == cluster

    plt.scatter(
        X_tsne[mask, 0],
        X_tsne[mask, 1],
        color=orange_shades[cluster - 1],
        s=35,
        alpha=0.9,
        edgecolors='#333333',
        linewidths=0.5
    )

plt.title(
    "Type 1 t-SNE",
    fontsize=28,
    fontweight="bold",
    pad=15
)

style_axes()

plt.savefig(
    "tSNE_Type1.png",
    dpi=300,
    bbox_inches='tight'
)

plt.show()

plt.figure(figsize=(8, 8))

cluster_list = list(range(7, 31))

for idx, cluster in enumerate(cluster_list):

    shade_index = idx % 8
    marker_index = idx // 8

    color = green_base_shades[shade_index]
    marker = markers[marker_index]

    mask = clusters_30 == cluster

    if marker == 'x':

        plt.scatter(
            X_tsne[mask, 0],
            X_tsne[mask, 1],
            color=color,
            marker=marker,
            s=40,
            linewidths=1.2
        )

    else:

        plt.scatter(
            X_tsne[mask, 0],
            X_tsne[mask, 1],
            color=color,
            marker=marker,
            s=35,
            alpha=0.9,
            edgecolors='#333333',
            linewidths=0.5
        )

plt.title(
    "Type 2 t-SNE",
    fontsize=28,
    fontweight="bold",
    pad=15
)

style_axes()

plt.savefig(
    "tSNE_Type2.png",
    dpi=300,
    bbox_inches='tight'
)

plt.show()

legend_elements = []

for cluster in range(1, 7):

    legend_elements.append(
        Line2D(
            [0], [0],
            marker='o',
            color='w',
            label=cluster_label_map[cluster],
            markerfacecolor=orange_shades[cluster - 1],
            markeredgecolor='#333333',
            markersize=11
        )
    )

cluster_list = list(range(7, 31))

for idx, cluster in enumerate(cluster_list):

    color = green_base_shades[idx % 8]
    marker = markers[idx // 8]

    if marker == 'x':

        legend_elements.append(
            Line2D(
                [0], [0],
                marker=marker,
                color=color,
                label=cluster_label_map[cluster],
                markeredgewidth=2,
                markersize=11,
                linestyle='None'
            )
        )

    else:

        legend_elements.append(
            Line2D(
                [0], [0],
                marker=marker,
                color='w',
                label=cluster_label_map[cluster],
                markerfacecolor=color,
                markeredgecolor='#333333',
                markersize=11
            )
        )

fig = plt.figure(figsize=(18, 6))

plt.legend(
    handles=legend_elements,
    loc='center',
    ncol=5,
    frameon=True,
    edgecolor='black',
    fontsize=12,
    title="Clusters",
    title_fontsize=14,
    columnspacing=1.8,
    handletextpad=0.6
)

plt.axis('off')

plt.savefig(
    "Legend.png",
    dpi=300,
    bbox_inches='tight'
)

plt.show()